# Problem 83: Agent Handoff with Context

**Difficulty:** Medium | **Framework:** LangGraph

**Categories:** multi-agent, memory

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/agent-handoff-with-context).

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The full conversation history must be preserved during handoff
- Agent B must be aware of what Agent A discussed with the user
- You may not ask the user to repeat information after a handoff
- The handoff mechanism must be explicit, not implicit


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

llm = ChatOpenAI(model="gpt-4o-mini")

class State(TypedDict):
    user_message: str
    sales_response: str
    onboarding_response: str

def sales_node(state: State) -> dict:
    prompt = f"You are a sales agent. The customer says: {state['user_message']}"
    response = llm.invoke(prompt)
    return {"sales_response": response.content}

def onboarding_node(state: State) -> dict:
    # BUG: No context from sales conversation is passed
    # TODO: Include the sales conversation in the prompt
    prompt = f"You are an onboarding specialist. The customer says: Set up my account now."
    response = llm.invoke(prompt)
    return {"onboarding_response": response.content}

graph = StateGraph(State)
graph.add_node("sales", sales_node)
graph.add_node("onboarding", onboarding_node)
graph.add_edge(START, "sales")
graph.add_edge("sales", "onboarding")
graph.add_edge("onboarding", END)

app = graph.compile()
result = app.invoke({"user_message": "I'm interested in the Pro plan for my team of 10", "sales_response": "", "onboarding_response": ""})
print("Onboarding:", result["onboarding_response"])

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Pass the conversation history as part of the handoff payload to Agent B
2. In OpenAI Agents SDK, handoffs automatically carry context — make sure you're using them correctly
3. In LangGraph, store conversation messages in the state so they persist across node transitions


## 7. Evaluation Criteria

Check your solution against these criteria:

- Conversation context preserved
- No repeated questions
- Seamless handoff experience
- Explicit handoff mechanism
